# Python Lab: Hypothesis Testing

**Libraries:** `pandas`, `seaborn`, and `scipy.stats`

Run the cells from top to bottom. The notebook uses the `tips`, `exercise`, and `titanic` datasets available through Seaborn.

In [1]:
import pandas as pd
import seaborn as sns
from scipy import stats

## Problem 1: One-Sample T-Test

**Hypotheses**

- H₀: The population mean total bill is $18.00 (μ = 18).
- Hₐ: The population mean total bill differs from $18.00 (μ ≠ 18).

Significance level: α = 0.05.

In [2]:
tips = sns.load_dataset("tips")

t_statistic, p_value = stats.ttest_1samp(tips["total_bill"], popmean=18)

print("t-statistic:", t_statistic)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0: the average total bill differs significantly from $18.00.")
else:
    print("Fail to reject H0: there is not enough evidence that the average total bill differs from $18.00.")

t-statistic: 3.1336805941629344
p-value: 0.0019383686765825567
Reject H0: the average total bill differs significantly from $18.00.


## Problem 2: Independent Two-Sample T-Test (Lunch vs Dinner)

**Hypotheses**

- H₀: The mean tip amount is the same for lunch and dinner.
- Hₐ: The mean tip amounts differ between lunch and dinner.

First, Levene’s test checks whether the group variances can be treated as equal. We then set `equal_var` based on that result.

In [3]:
lunch_tips = tips[tips["time"] == "Lunch"]["tip"]
dinner_tips = tips[tips["time"] == "Dinner"]["tip"]

levene_statistic, levene_p_value = stats.levene(lunch_tips, dinner_tips)

print("Levene's test statistic:", levene_statistic)
print("Levene's test p-value:", levene_p_value)

equal_var = levene_p_value >= 0.05
print("Treat variances as equal:", equal_var)

t_statistic, p_value = stats.ttest_ind(lunch_tips, dinner_tips, equal_var=equal_var)

print("t-statistic:", t_statistic)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0: average tip amounts differ significantly between lunch and dinner.")
else:
    print("Fail to reject H0: there is not enough evidence of a difference in average tip amounts.")

Levene's test statistic: 0.88098601335503
Levene's test p-value: 0.3488676006910501
Treat variances as equal: True
t-statistic: -1.9062569301202392
p-value: 0.057801534751715615
Fail to reject H0: there is not enough evidence of a difference in average tip amounts.


## Problem 3: Paired T-Test (Pulse at 1 min vs 15 min)

The same participants are measured at both times, so a paired t-test is appropriate.

**Hypotheses**

- H₀: The mean paired pulse difference is zero.
- Hₐ: The mean paired pulse difference is not zero.

In [4]:
exercise = sns.load_dataset("exercise")

pulse_data = exercise[exercise["time"].isin(["1 min", "15 min"])].pivot(
    index="id", columns="time", values="pulse"
).dropna()

pulse_1_min = pulse_data["1 min"]
pulse_15_min = pulse_data["15 min"]

t_statistic, p_value = stats.ttest_rel(pulse_15_min, pulse_1_min)
average_difference = (pulse_15_min - pulse_1_min).mean()

print("t-statistic:", t_statistic)
print("p-value:", p_value)
print("Average pulse difference (15 min - 1 min):", average_difference)

if p_value < 0.05:
    print("Reject H0: pulse changed significantly between 1 minute and 15 minutes.")
else:
    print("Fail to reject H0: there is not enough evidence of a significant pulse change.")

t-statistic: 3.727066004148019
p-value: 0.0008348847676369906
Average pulse difference (15 min - 1 min): 8.4
Reject H0: pulse changed significantly between 1 minute and 15 minutes.


## Problem 4: Chi-Square Goodness of Fit Test

The expected proportions are 25% First Class, 25% Second Class, and 50% Third Class.

**Hypotheses**

- H₀: The Titanic passenger-class distribution follows the expected proportions.
- Hₐ: The distribution differs from the expected proportions.

In [5]:
titanic = sns.load_dataset("titanic")

class_order = ["First", "Second", "Third"]
observed_counts = titanic["class"].value_counts().reindex(class_order, fill_value=0).to_numpy()

total_passengers = observed_counts.sum()
expected_proportions = [0.25, 0.25, 0.50]
expected_counts = [total_passengers * proportion for proportion in expected_proportions]

print("Observed counts (First, Second, Third):", observed_counts)
print("Expected counts (First, Second, Third):", expected_counts)

chi_statistic, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print("Chi-square statistic:", chi_statistic)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0: the Titanic passenger-class distribution differs significantly from the historical proportions.")
else:
    print("Fail to reject H0: there is not enough evidence that the distribution differs from the historical proportions.")

Observed counts (First, Second, Third): [216 184 491]
Expected counts (First, Second, Third): [np.float64(222.75), np.float64(222.75), np.float64(445.5)]
Chi-square statistic: 11.592592592592592
p-value: 0.0030387887018808505
Reject H0: the Titanic passenger-class distribution differs significantly from the historical proportions.


## Final note

For all four tests, use **α = 0.05**. A p-value below 0.05 means reject the null hypothesis; otherwise, fail to reject it.